# 02. ACP (Agent Client Protocol) for DeepAgents

> **주제**: 코딩 에이전트를 *에디터/IDE* 와 표준 방식으로 연결하기 — `deepagents-acp`
>
> **원문**: https://docs.langchain.com/oss/python/deepagents/acp

---

## 이 노트북에서 배우는 것
1. ACP 가 무엇이고 **MCP 와 어떻게 다른가**
2. `deepagents-acp` 로 deep agent 를 ACP 서버로 노출하기
3. stdio 동작 방식과 지원 클라이언트(Zed, JetBrains, VS Code, Neovim)
4. Zed 연동 절차

## 1. ACP vs MCP — 헷갈리지 말 것

둘 다 "표준 프로토콜"이지만 **연결하는 대상이 다릅니다.**

| | **MCP** (Model Context Protocol) | **ACP** (Agent Client Protocol) |
|---|---|---|
| 연결 대상 | 에이전트 ↔ **외부 도구/데이터** | 에이전트 ↔ **코드 에디터/IDE** |
| 누가 호스트? | 에이전트가 도구를 *소비* | 에디터가 에이전트를 *호스팅* |
| 대표 예시 | DB, API, 파일시스템 도구 | Zed, JetBrains, VS Code 의 에이전트 패널 |

> 원문 표현: *"ACP is designed for agent-editor integrations"* — 즉 ACP 는 **에이전트-에디터 통합**을 위한 것이고, MCP 는 **외부 도구 통합**을 위한 것입니다.

```
[Zed / VS Code]  --ACP(stdio)-->  [Deep Agent]  --MCP-->  [외부 도구들]
      에디터                          에이전트              도구/데이터
```

In [ ]:
# 설치 (uv)
!uv pip install -q deepagents-acp
# uv 프로젝트라면:  uv add deepagents-acp

## 2. deep agent 를 ACP 서버로 노출하기

핵심은 세 줄입니다.
1. `create_deep_agent(...)` 로 에이전트를 만든다 (대화 이어가기용 `checkpointer` 권장)
2. `AgentServerACP(agent)` 로 감싼다
3. `run_agent(server)` 로 실행 → **stdio 모드**로 표준입력에서 요청을 읽고 표준출력으로 응답

> ⚠️ ACP 서버는 stdio 로 통신하므로, 보통 **노트북이 아니라 `.py` 파일**로 만들어 에디터가 서브프로세스로 실행하게 합니다. 아래 셀은 그 파일을 만들어 둡니다.

In [ ]:
acp_server = '''
import asyncio

from acp import run_agent
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver

from deepagents_acp.server import AgentServerACP


async def main() -> None:
    agent = create_deep_agent(
        model="google_genai:gemini-3.5-flash",
        system_prompt="You are a helpful coding assistant",
        checkpointer=MemorySaver(),
    )

    server = AgentServerACP(agent)
    await run_agent(server)


if __name__ == "__main__":
    asyncio.run(main())
'''

with open("acp_server.py", "w") as f:
    f.write(acp_server)
print("acp_server.py 생성 완료 — 에디터가 이 파일을 실행하도록 등록한다")

### 코드 뜯어보기

- `create_deep_agent(...)` — deepagents 의 메인 팩토리. `model`, `system_prompt`, `tools`, `subagents` 등을 받음
- `checkpointer=MemorySaver()` — 멀티턴 대화에서 **상태를 보존**. ACP 세션이 이어지려면 사실상 필수
- `AgentServerACP(agent)` — LangGraph 에이전트를 ACP 프로토콜 어댑터로 감쌈
- `await run_agent(server)` — stdio 루프 시작 (요청 수신 → 에이전트 실행 → 응답 송신)

## 3. 지원 클라이언트

ACP 호환 에디터에서 위 서버를 곧바로 붙일 수 있습니다.

- **Zed** (네이티브 지원)
- **JetBrains IDEs**
- **Visual Studio Code** (`vscode-acp` 확장 경유)
- **Neovim** (ACP 호환 플러그인 경유)

## 4. Zed 연동 절차

1. 레포지토리를 클론하고 의존성을 설치한다
2. `.env.example` 을 `.env` 로 복사하고 `ANTHROPIC_API_KEY` 를 설정한다
3. Zed 의 `settings.json` 의 `agent_servers` 아래에 서버 실행 커맨드를 등록한다
4. Zed 의 **Agents 패널**에서 에이전트를 사용한다

아래는 `settings.json` 등록 예시입니다 (실제 키 이름/경로는 deepagents 레포의 데모 엔트리포인트를 따르세요).

In [ ]:
# Zed settings.json 등록 예시 (JSON — 실제로는 ~/.config/zed/settings.json 에 작성)
zed_settings_example = '''
{
  "agent_servers": {
    "My Deep Agent": {
      "command": "python",
      "args": ["/absolute/path/to/acp_server.py"],
      "env": {
        "ANTHROPIC_API_KEY": "sk-ant-..."
      }
    }
  }
}
'''
print(zed_settings_example)

## 정리 & 연습 문제

**핵심 요약**
- ACP = **에이전트 ↔ 에디터** 통합 표준 (MCP 는 에이전트 ↔ 도구)
- `create_deep_agent` → `AgentServerACP` → `run_agent` 3단계로 stdio 서버 완성
- `checkpointer` 로 멀티턴 대화 상태 유지
- Zed/JetBrains/VS Code/Neovim 에서 서브프로세스로 등록해 사용

**연습**
1. `acp_server.py` 의 모델을 Anthropic 모델로 바꾸고 `system_prompt` 를 "파이썬 리뷰 전문가"로 수정해 보세요.
2. MCP 도구(`langchain-mcp-adapters`)를 `create_deep_agent(tools=...)` 로 붙여, ACP 로 노출되는 에이전트가 외부 도구도 쓰게 만들어 보세요. → MCP+ACP 조합!
3. (선택) Zed 또는 VS Code(vscode-acp)에 등록해 실제 에디터 패널에서 호출해 보세요.